# Ejercicio 2 - Buscar con palabras, no con numeros

**La unica diferencia con el ejercicio 1:** los vectores ya no los inventamos
nosotros. Los calcula un **modelo de embeddings** a partir del texto.

Resultado: ahora buscamos **escribiendo palabras**, no dando vectores.

## 1. Instalar dependencias

Sumamos `sentence-transformers`: el modelo que convierte texto en vector.

In [1]:
!pip install -q chromadb sentence-transformers 2>&1 | tail -n 2
import chromadb
from chromadb.utils import embedding_functions
print("Listo.")

google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Listo.


## 2. Preparar Chroma y el modelo de embeddings

El modelo `paraphrase-multilingual-MiniLM-L12-v2` convierte cualquier texto
en un vector de **384 numeros**. Es multilingue (es, en, pt, fr).

In [2]:
import os
RUTA = "/content/bd_con_modelo"
os.makedirs(RUTA, exist_ok=True)

cliente = chromadb.PersistentClient(path=RUTA)

# El modelo tarda unos segundos la primera vez (lo descarga de Hugging Face).
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)
print("Modelo cargado.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado.


## 3. Crear la coleccion (ahora con `embedding_function`)

Aqui esta el cambio clave frente al ejercicio 1: al crear la coleccion le
pasamos `embedding_function=ef`. A partir de ahora, **solo le damos texto**;
Chroma calcula el vector solo.

In [3]:
try:
    cliente.delete_collection("conceptos")
except Exception:
    pass

coleccion = cliente.create_collection(
    name="conceptos",
    embedding_function=ef,                  # <-- la novedad
    metadata={"hnsw:space": "cosine"},
)
print(f"Coleccion '{coleccion.name}' creada.")

Coleccion 'conceptos' creada.


## 4. Cargar datos (solo texto, sin vectores)

Datos pequeno: 12 palabras en 3 categorias. Nota que **no** pasamos
`embeddings`, solo `documents`. Chroma los calcula con el modelo.

In [8]:
coleccion.upsert(
    ids=["a1","a2","a3","a4","a5",                # animales
         "v1","v2","v3","v4",                     # vehiculos
         "c1","c2","c3"],                         # comida
    documents=["perro","gato","caballo","lobo","leon",
               "carro","bicicleta","avion","barco",
               "pan","manzana","leche"],
    metadatas=[{"tipo":"animal"}]*5 +
              [{"tipo":"vehiculo"}]*4 +
              [{"tipo":"comida"}]*3,
)
print(f"Registros guardados: {coleccion.count()}")

Registros guardados: 12


## 5. Buscar con una palabra

Le damos **texto** a `query_texts`. Chroma vectoriza la consulta con el mismo
modelo y busca los vecinos mas cercanos.

In [9]:
def buscar(q, n=5):
    r = coleccion.query(query_texts=[q], n_results=n)
    print(f"Consulta: '{q}'\n")
    for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0]):
        sim = round((1 - dist) * 100, 1)
        print(f"  {doc:<12} [{meta['tipo']:<8}] similitud: {sim}%")
    print()

# Caso 1: una palabra que existe.
buscar("perro")

Consulta: 'perro'

  perro        [animal  ] similitud: 100.0%
  lobo         [animal  ] similitud: 65.0%
  caballo      [animal  ] similitud: 53.2%
  pan          [comida  ] similitud: 46.3%
  leche        [comida  ] similitud: 45.9%



^Ves lo que paso? Al buscar `perro`, los primeros resultados no son solo
`perro`: tambien aparecen **gato, lobo, leon, caballo**. El modelo entiende
que son **animales** y los ubica cerca en el espacio de vectores.

## 6. La magia: buscar sin palabras en comun

Esto no se podia en una BD normal. Probemos descripciones que **no contienen
ninguna palabra** de los datos guardados.

In [13]:
# "mamifero que ladra" -> no contiene 'perro' ni 'lobo', pero...
buscar("mamífero que ladra")

# "vehiculo para volar" -> no contiene 'avion'...
buscar("vehículo para volar")

# "alimento hecho de harina" -> no contiene 'pan'...
buscar("alimento hecho de harina")

Consulta: 'mamífero que ladra'

  lobo         [animal  ] similitud: 71.0%
  perro        [animal  ] similitud: 65.3%
  caballo      [animal  ] similitud: 61.0%
  gato         [animal  ] similitud: 52.4%
  pan          [comida  ] similitud: 47.2%

Consulta: 'vehículo para volar'

  avion        [vehiculo] similitud: 70.5%
  carro        [vehiculo] similitud: 54.2%
  barco        [vehiculo] similitud: 40.1%
  pan          [comida  ] similitud: 34.0%
  leon         [animal  ] similitud: 27.9%

Consulta: 'alimento hecho de harina'

  manzana      [comida  ] similitud: 48.3%
  leche        [comida  ] similitud: 41.8%
  pan          [comida  ] similitud: 29.1%
  lobo         [animal  ] similitud: 19.9%
  caballo      [animal  ] similitud: 16.7%



## 7. Comparar con busqueda exacta (LIKE)

Para que se note el contraste, la misma consulta por `LIKE` no devuelve nada:

In [14]:
# Equivalente a WHERE document ILIKE '%mamifero que ladra%'
q = "mamífero que ladra".lower()
exactos = [d for d in ["perro","gato","caballo","lobo","leon",
                       "carro","bicicleta","avion","barco",
                       "pan","manzana","leche"]
           if q in d.lower()]
print(f"Busqueda exacta '{q}': {exactos or 'SIN RESULTADOS'}")

Busqueda exacta 'mamífero que ladra': SIN RESULTADOS


## 8. Cierre

- En el ejercicio 1 **inventabamos** los vectores (2 numeros, a mano).
- Ahora **el modelo los calcula** (384 numeros, a partir del texto).
- Por eso podemos **escribir palabras** y encontrar lo parecido, incluso sin
  palabras en comun.

### Siguiente paso (ejercicio 3)

Lo mismo, pero con **mas datos** (148 registros en 4 idiomas), filtro por
metadatos (idioma, categoria) y una mirada al indice **HNSW**: como busca
rapido sin comparar contra todos.

In [21]:
r = coleccion.get(ids=["a1"], include=["documents", "embeddings"])
print(r["documents"][0], "->", r["embeddings"][0])

perro -> [ 0.20481233 -0.17759715 -0.18733425  0.14696909 -0.15670754 -0.06387013
 -0.11809503  0.21284693  0.30482218 -0.23305626 -0.00869941 -0.24155715
  0.04219599 -0.2831631  -0.3554118  -0.09708956 -0.07078849  0.00911475
 -0.05706539 -0.06398008  0.08880333  0.03924305  0.11921379 -0.16597465
 -0.42870757 -0.24445984 -0.08034892  0.0263088  -0.0110047  -0.22180894
  0.25991124 -0.20421422  0.0390043  -0.2346198  -0.04475446  0.16837302
  0.40715659 -0.27678519 -0.13782755  0.11466521  0.37345448  0.03298514
  0.27749726  0.01864212 -0.29047143  0.01442802 -0.3592855  -0.36309478
 -0.14744112 -0.12632495  0.33135557  0.2631247  -0.06280491  0.16677174
  0.36556539  0.18230064  0.46663201  0.44162738 -0.14045165  0.29355332
 -0.15179552  0.19569664 -0.36946103  0.285092    0.02192513 -0.11811347
 -0.50021517 -0.1686738  -0.47974554  0.34757417  0.18928742  0.00957764
 -0.08792004  0.13822776  0.30385581  0.11625464 -0.07761981  0.15097752
  0.02471625  0.10940095  0.08060171  0.12